# 11 — BERTopic Checkpoints: Distance Bands & Year Slices

Fits **10 independent BERTopic models** and saves each as a `.pkl` checkpoint:

| Run ID | Slice |
| --- | --- |
| `coast_band_A` | `distance2coastline < 0.1 km` (beachfront) |
| `coast_band_B` | `0.1 ≤ distance2coastline < 0.5 km` (near-coast) |
| `coast_band_C` | `distance2coastline ≥ 0.5 km` (inland) |
| `year_2018` … `year_2024` | one model per review year |

Results (topic assignments + top-word tables) are written to `TOPIC_LABELS` and `REVIEW_TOPICS` in `hotel_reviews.db`.
LLM labelling is done separately in the next notebook.

**Prerequisites**
```bash
uv run python src/preprocess_to_duckdb.py   # REVIEW_TEXT_PROCESSED
uv run python src/embed_to_duckdb.py        # REVIEW_EMBEDDINGS
```

## Section 1 — Imports & Config

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import pickle
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from src.topic_modeling import load_from_duckdb, build_bertopic

# ── Paths ─────────────────────────────────────────────────────────────────
DB_PATH   = Path("../data/hotel_reviews.db")
CKPT_DIR  = Path("../checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Distance-band slices ──────────────────────────────────────────────────
BAND_SLICES = {
    "coast_band_A": "r.distance2coastline < 0.1",
    "coast_band_B": "r.distance2coastline >= 0.1 AND r.distance2coastline < 0.5",
    "coast_band_C": "r.distance2coastline >= 0.5",
}

# ── Year slices ───────────────────────────────────────────────────────────
YEAR_SLICES = {f"year_{y}": y for y in range(2018, 2025)}

# ── BERTopic hyperparams ──────────────────────────────────────────────────
# Reduce min_cluster_size for small slices (<10k rows) automatically
DEFAULT_MIN_CLUSTER = 50
SMALL_MIN_CLUSTER   = 20   # used when slice < 10_000 rows
SMALL_THRESHOLD     = 10_000

print(f"DB          : {DB_PATH.resolve()}")
print(f"Checkpoints : {CKPT_DIR.resolve()}")
print(f"Runs planned: {len(BAND_SLICES) + len(YEAR_SLICES)}")

## Section 2 — Create DuckDB Tables

`TOPIC_LABELS` — one row per (run_id, topic_id): stores top words and (later) LLM seed-topic label.  
`REVIEW_TOPICS` — one row per (run_id, review_id): stores topic assignment and probability.

In [ ]:
con = duckdb.connect(str(DB_PATH))

con.execute("""
    CREATE TABLE IF NOT EXISTS TOPIC_LABELS (
        run_id     VARCHAR NOT NULL,
        topic_id   INTEGER NOT NULL,
        top_words  VARCHAR,
        n_docs     INTEGER,
        seed_topic VARCHAR,
        seed_score FLOAT,
        PRIMARY KEY (run_id, topic_id)
    )
""")

con.execute("""
    CREATE TABLE IF NOT EXISTS REVIEW_TOPICS (
        run_id    VARCHAR  NOT NULL,
        review_id VARCHAR  NOT NULL,
        topic_id  INTEGER  NOT NULL,
        prob      FLOAT,
        PRIMARY KEY (run_id, review_id)
    )
""")

con.close()
print("Tables ready: TOPIC_LABELS, REVIEW_TOPICS")

## Section 3 — Helper: Write BERTopic Results to DuckDB

In [ ]:
import csv

def fit_and_save(run_id: str, docs: list, embeddings, min_cs: int, ckpt: Path):
    """Fit BERTopic and pickle the model + hard topic assignments.

    Returns (topic_model, topics).  Probabilities are never computed.
    """
    from src.topic_modeling import build_bertopic

    print(f"  Fitting BERTopic (min_cluster_size={min_cs}, min_samples=5) …")
    topic_model = build_bertopic(
        nr_topics="auto",
        min_cluster_size=min_cs,
        min_topic_size=min_cs,
    )
    topics, _ = topic_model.fit_transform(docs, embeddings)

    with open(ckpt, "wb") as f:
        pickle.dump({"model": topic_model, "topics": topics}, f)
    print(f"  Checkpoint saved → {ckpt.name}")
    return topic_model, topics


def write_to_duckdb(run_id: str, topic_model, df: pd.DataFrame, topics: list, db_path: Path):
    """Write hard topic assignments and top-word labels to DuckDB."""
    con = duckdb.connect(str(db_path))

    # TOPIC_LABELS — one row per discovered topic (including outlier -1)
    topic_info = topic_model.get_topic_info()
    label_rows = []
    for _, row in topic_info.iterrows():
        tid = int(row["Topic"])
        kw_list = topic_model.get_topic(tid)
        top_words = ", ".join(w for w, _ in kw_list[:10]) if kw_list else ""
        label_rows.append((run_id, tid, top_words, int(row["Count"]), None, None))

    con.executemany(
        "INSERT OR IGNORE INTO TOPIC_LABELS "
        "(run_id, topic_id, top_words, n_docs, seed_topic, seed_score) "
        "VALUES (?, ?, ?, ?, ?, ?)",
        label_rows,
    )

    # REVIEW_TOPICS — one row per review (hard assignment, prob=NULL)
    review_rows = [
        (run_id, rid, int(tid), None)
        for rid, tid in zip(df["review_id"].tolist(), topics)
    ]
    con.executemany(
        "INSERT OR IGNORE INTO REVIEW_TOPICS (run_id, review_id, topic_id, prob) "
        "VALUES (?, ?, ?, ?)",
        review_rows,
    )

    con.close()
    n_topics = sum(1 for r in label_rows if r[1] != -1)
    print(f"  [{run_id}] DuckDB: {n_topics} topics, {len(review_rows):,} review rows")


def export_topic_csv(run_id: str, topic_model, df: pd.DataFrame, topics: list, out_dir: Path):
    """Export two CSVs per run:
    - <run_id>_topic_info.csv   : topic_id, n_docs, top_words, representative docs
    - <run_id>_review_topics.csv: review_id, hotel_id, review_year, topic_id
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    # topic info
    topic_info = topic_model.get_topic_info().copy()
    topic_info.insert(0, "run_id", run_id)
    topic_info_path = out_dir / f"{run_id}_topic_info.csv"
    topic_info.to_csv(topic_info_path, index=False, encoding="utf-8-sig")

    # per-review assignment (no probs)
    result = df[["review_id", "hotel_id", "review_year", "language"]].copy()
    result["topic_id"] = topics
    result.insert(0, "run_id", run_id)
    review_path = out_dir / f"{run_id}_review_topics.csv"
    result.to_csv(review_path, index=False, encoding="utf-8-sig")

    print(f"  [{run_id}] CSVs → {topic_info_path.name}, {review_path.name}")


print("Helpers defined: fit_and_save | write_to_duckdb | export_topic_csv")

## Section 4 — Run: Distance-Band Models

Three BERTopic fits (beachfront / near-coast / inland).  
Each checkpoint is skipped if `checkpoints/<run_id>.pkl` already exists.

In [ ]:
CSV_DIR = Path("../data/topic_results")

for run_id, where_clause in BAND_SLICES.items():
    ckpt = CKPT_DIR / f"{run_id}.pkl"
    print(f"\n{'='*60}")
    print(f"Run    : {run_id}")
    print(f"Filter : {where_clause}")

    df, docs, embeddings = load_from_duckdb(
        db_path=DB_PATH,
        extra_where=f"{where_clause} AND r.distance2coastline IS NOT NULL",
    )
    print(f"  Rows : {len(docs):,}")

    min_cs = SMALL_MIN_CLUSTER if len(docs) < SMALL_THRESHOLD else DEFAULT_MIN_CLUSTER

    if ckpt.exists():
        print(f"  Checkpoint found — loading {ckpt.name}")
        with open(ckpt, "rb") as f:
            saved = pickle.load(f)
        topic_model = saved["model"]
        topics      = saved["topics"]
    else:
        topic_model, topics = fit_and_save(run_id, docs, embeddings, min_cs, ckpt)

    n_topics  = len(set(topics)) - (1 if -1 in topics else 0)
    n_outlier = sum(1 for t in topics if t == -1)
    print(f"  Topics: {n_topics}  |  Outliers: {n_outlier:,} ({n_outlier/len(topics)*100:.1f}%)")

    write_to_duckdb(run_id, topic_model, df, topics, DB_PATH)
    export_topic_csv(run_id, topic_model, df, topics, CSV_DIR)

print("\nDistance-band runs complete.")

## Section 5 — Run: Year-Slice Models (2018–2024)

Seven BERTopic fits, one per year. Years with < 10 k reviews automatically use a smaller `min_cluster_size`.

In [ ]:
for run_id, year in YEAR_SLICES.items():
    ckpt = CKPT_DIR / f"{run_id}.pkl"
    print(f"\n{'='*60}")
    print(f"Run    : {run_id}  (year={year})")

    df, docs, embeddings = load_from_duckdb(
        db_path=DB_PATH,
        min_year=year,
        max_year=year,
    )
    print(f"  Rows : {len(docs):,}")

    min_cs = SMALL_MIN_CLUSTER if len(docs) < SMALL_THRESHOLD else DEFAULT_MIN_CLUSTER

    if ckpt.exists():
        print(f"  Checkpoint found — loading {ckpt.name}")
        with open(ckpt, "rb") as f:
            saved = pickle.load(f)
        topic_model = saved["model"]
        topics      = saved["topics"]
    else:
        topic_model, topics = fit_and_save(run_id, docs, embeddings, min_cs, ckpt)

    n_topics  = len(set(topics)) - (1 if -1 in topics else 0)
    n_outlier = sum(1 for t in topics if t == -1)
    print(f"  Topics: {n_topics}  |  Outliers: {n_outlier:,} ({n_outlier/len(topics)*100:.1f}%)")

    write_to_duckdb(run_id, topic_model, df, topics, DB_PATH)
    export_topic_csv(run_id, topic_model, df, topics, CSV_DIR)

print("\nYear-slice runs complete.")

## Section 6 — Verify Checkpoint Inventory

In [ ]:
all_run_ids = list(BAND_SLICES.keys()) + list(YEAR_SLICES.keys())

rows = []
for run_id in all_run_ids:
    ckpt = CKPT_DIR / f"{run_id}.pkl"
    size_mb = f"{ckpt.stat().st_size / 1e6:.1f} MB" if ckpt.exists() else "MISSING"
    rows.append({"run_id": run_id, "checkpoint": ckpt.name, "size": size_mb})

inventory = pd.DataFrame(rows)
print(inventory.to_string(index=False))

## Section 7 — Verify DuckDB Tables

In [ ]:
con = duckdb.connect(str(DB_PATH), read_only=True)

print("=== TOPIC_LABELS — topics per run ===")
tl = con.execute("""
    SELECT run_id,
           COUNT(*) FILTER (WHERE topic_id != -1) AS n_topics,
           SUM(n_docs) FILTER (WHERE topic_id != -1) AS docs_in_topics,
           SUM(n_docs) FILTER (WHERE topic_id  = -1) AS outliers
    FROM TOPIC_LABELS
    GROUP BY run_id
    ORDER BY run_id
""").df()
print(tl.to_string(index=False))

print("\n=== REVIEW_TOPICS — assignments per run ===")
rt = con.execute("""
    SELECT run_id, COUNT(*) AS n_reviews
    FROM REVIEW_TOPICS
    GROUP BY run_id
    ORDER BY run_id
""").df()
print(rt.to_string(index=False))

con.close()

## Section 8 — Preview Top Words per Run

Quick sanity check — shows the 5 largest topics (excluding outlier -1) for each run.

In [ ]:
con = duckdb.connect(str(DB_PATH), read_only=True)

preview = con.execute("""
    SELECT run_id, topic_id, n_docs, top_words
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY run_id ORDER BY n_docs DESC) AS rn
        FROM TOPIC_LABELS
        WHERE topic_id != -1
    ) t
    WHERE rn <= 5
    ORDER BY run_id, n_docs DESC
""").df()

con.close()

for run_id, grp in preview.groupby("run_id"):
    print(f"\n── {run_id} ──")
    for _, row in grp.iterrows():
        print(f"  topic {row.topic_id:>3}  ({row.n_docs:>6} docs)  {row.top_words}")